In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
%config InlineBackend.figure_format = 'retina'

In [ ]:
import os, sys
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    # sys.path.append(parent_dir)
    sys.path.insert(0, parent_dir)
    print(f"{parent_dir} added to sys.path")
    
from ib_insync import IB, MarketOrder, LimitOrder, BarData, BarDataList, Stock, util

In [ ]:
import os
import datetime
import pickle
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format=('%(asctime)s:' + logging.BASIC_FORMAT))
global logger
logger = logging.getLogger(__name__)


In [ ]:
dtnow = datetime.datetime.now() + datetime.timedelta(days=-4)
sym = 'NVDA'
barsize = '1 min'
cachefilename = f'data/{sym}_{barsize.replace(' ', '_')}_data_{dtnow:%y%m%d}.pkl'
logger.info(f"Cache file: {cachefilename}")

In [ ]:
logger.info("Using cached data...")
if os.path.exists(cachefilename):
    with open(cachefilename, 'rb') as f:
        data = pickle.load(f)
        logger.info(f"Loaded {len(data)} symbols")
        # ekHistDataInitEnd = eventkit.Event('barsInitEnd')
        # ekHistDataInitEnd += onHistDataEnd
        # ekHistData = eventkit.Event('bars')
        # ekHistData += onBarUpdate
        bars1m = BarDataList()
        for sym, bars in data.items():
            logger.info(f"Processing {sym}...")
            for bar in bars:
                bars1m.append(bar)
                # ekHistData.emit(bars1m, True)
                # prev_avg, cur_avg, peak, valley = pkvl2.process_price(bar.average, bar.date, method=2)
                # if peak:
                #     logger.info(f"peak detected: {peak[0]:.2f} {peak[1]:%H:%M:%S}")
                # if valley:
                #     logger.info(f"valley detected: {valley[0]:.2f} {valley[1]:%H:%M:%S}")
        logger.info(f"Processing {len(bars1m)} bars...")
    logger.info("Done processing cached data.")
else:
    logger.info(f"Cache file not found: {cachefilename}")


In [ ]:
df = util.df(bars1m)

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

# Initialize starting price and number of minutes in the trading day
starting_price = 100.0
num_minutes = (16 * 60) - (9 * 60 + 30)  # Total trading minutes from 9:30 AM to 4:00 PM

useRandomPrice = False
if useRandomPrice:
    # Generate random price changes
    np.random.seed(42)  # For reproducibility
    price_changes = np.random.normal(0, 1, num_minutes)  # Mean 0, standard deviation 1

    # Simulate prices
    prices = starting_price + np.cumsum(price_changes)
else:
    prices = [bar.average for bar in bars1m]

def get_current_price(minute):
    return prices[minute - (9 * 60 + 30)]  # Adjust index to start at 9:30 AM

# # Example usage
# for minute in range(9 * 60 + 30, 9 * 60 + 35):  # First 5 minutes
#     print(f"Minute {minute}: ${get_current_price(minute):.2f}")


In [ ]:
df['logret'] = np.log(df['average'] / df['average'].shift(1))

In [ ]:
df['logret'].iloc[:50].dropna().values

In [ ]:
trading_start_time = 9 * 60 + 30
trading_end_time = 16 * 60

In [ ]:
# Initialize variables to store the previous and current prices
prev_price = None
current_price = None
peak = None
valley = None

# Initialize lists to store peaks and valleys
peaks = []
valleys = []

# Define a function to process each incoming price
def process_price(price, prev_price, current_price, peak, valley, peaks, valleys, minute):
    prev_price = current_price
    current_price = price
    
    # Check for peak
    if prev_price and current_price > prev_price:
        if peak is None or current_price > peak[0]:
            peak = (current_price, minute)
        if valley:
            valleys.append(valley)
            valley = None

    # Check for valley
    if prev_price and current_price < prev_price:
        if valley is None or current_price < valley[0]:
            valley = (current_price, minute)
        if peak:
            peaks.append(peak)
            peak = None
    
    return prev_price, current_price, peak, valley

def process_price2(price, prev_price, current_price, peak, valley, peaks, valleys, minute):
    prev_price = current_price
    current_price = price
    
    # Check for peak
    if prev_price is not None and current_price > prev_price:
        if peak is None or current_price > peak:
            peak = current_price
        if valley:
            valleys.append((valley, minute))
            valley = None
    elif prev_price is not None and current_price < prev_price:
        if peak:
            peaks.append(peak)
            peak = None

    # Check for valley
    if prev_price is not None and current_price < prev_price:
        if valley is None or current_price < valley:
            valley = current_price
        if peak:
            peaks.append(peak)
            peak = None
    elif prev_price is not None and current_price > prev_price:
        if valley:
            valleys.append(valley)
            valley = None
    
    return prev_price, current_price, peak, valley

# Simulate the trading day minute by minute
fig, ax = plt.subplots(figsize=(50, 6))
price_lst = []
for minute in range(trading_start_time, trading_end_time):
    price = get_current_price(minute)
    price_lst.append(price)
    prev_price, current_price, peak, valley = process_price(
        price, prev_price, current_price, peak, valley, peaks, valleys, minute)
    # # plot the prices and peaks and valleys
    # if peak:
    #     ax.plot(minute, peak, 'ro')
    # if valley:
    #     ax.plot(minute, valley, 'bo')
    # ax.plot(minute, price, 'g')
    # plt.show()

# plot the prices and peaks and valleys
ax.plot(range(trading_start_time, trading_end_time), price_lst)
for peak in peaks:
    ax.plot(peak[1], peak[0], 'ro')
for valley in valleys:
    ax.plot(valley[1], valley[0], 'bo')
# plt.plot(range(trading_start_time, trading_end_time), peaks, 'ro')
# plt.plot(range(trading_start_time, trading_end_time), valleys, 'bo')
plt.show()

# Output the recorded peaks and valleys
print("Peaks:", peaks)
print("Valleys:", valleys)


In [ ]:
class PriceDetector:
    def __init__(self):
        self.prev_price = None
        current_price = None
        peak = None
        self.valley = None
        self.peaks = []
        self.valleys = []
        self.r = []

    def process_price(self, price, minute):
        self.prev_price = self.current_price
        self.current_price = price
        
        if self.prev_price is not None:
            self.r.append(np.log(self.current_price / self.prev_price))
        else:
            self.r.append(0)
        r = self.r[0:]
        logger.info(f"{np.average(r[-5:]):.2%} {np.average(r[-4:]):.2%} {np.average(r[-3:]):.2%} {np.average(r[-2:]):.2%} {r[-1:][0]:.2%}")

        if self.prev_price is not None:
            if self.current_price > self.prev_price:
                # self._handle_price_increase(minute)
                if self.peak is None or self.current_price > self.peak[0]:
                    self.peak = (self.current_price, minute)
                if self.valley is not None:
                    self.valleys.append(self.valley)
                    self.valley = None
            elif self.current_price < self.prev_price:
                # self._handle_price_decrease(minute)
                if self.valley is None or self.current_price < self.valley[0]:
                    self.valley = (self.current_price, minute)
                if self.peak is not None:
                    self.peaks.append(self.peak)
                    self.peak = None
        
        return self.prev_price, self.current_price, self.peak, self.valley

    # def _handle_price_increase(self, minute):
    #     if self.peak is None or self.current_price > self.peak:
    #         self.peak = self.current_price
    #     if self.valley is not None:
    #         self.valleys.append((self.valley, minute))
    #         self.valley = None

    # def _handle_price_decrease(self, minute):
    #     if self.valley is None or self.current_price < self.valley:
    #         self.valley = self.current_price
    #     if self.peak is not None:
    #         self.peaks.append(self.peak)
    #         self.peak = None

    def get_peaks(self):
        return self.peaks

    def get_valleys(self):
        return self.valleys


In [ ]:
import inspect
import filterpy
inspect.getfile(filterpy)
from filterpy.kalman import KalmanFilter, UnscentedKalmanFilter, JulierSigmaPoints, unscented_transform, MerweScaledSigmaPoints, IMMEstimator
from filterpy.common import Q_discrete_white_noise, Q_continuous_white_noise, Saver

def make_ca_filter(dt, std_R):
    cafilter = KalmanFilter(dim_x=3, dim_z=1)
    # cafilter.x = np.array([0., 0., 0.])
    cafilter.P *= 3
    cafilter.R *= std_R*std_R
    cafilter.Q = Q_discrete_white_noise(dim=3, dt=dt, var=0.01)
    cafilter.F = np.array([[1, dt, 0.5*dt*dt],
                           [0, 1,         dt], 
                           [0, 0,          1]])
    cafilter.H = np.array([[0, 1.0, 0]])
    return cafilter

dt_one = 1.
initial_price = 0.
kf_ca = make_ca_filter(dt_one, std_R=0.01)
kf_ca_saver = Saver(kf_ca)
kf_ca.x = np.array([initial_price, 0, 0]).T
kf_ca.P *= 0.01*0.01
kf_ca.R *= 0.01*0.01
kf_ca.Q = Q_continuous_white_noise(dim=3, dt=dt_one, spectral_density=1.)


In [ ]:
detector0 = PriceDetector()

price_lst2 = []
time_lst = []
base_price = get_current_price(trading_start_time)
for minute in range(trading_start_time, trading_start_time+30): # trading_end_time
    price = get_current_price(minute)
    price_lst2.append(price)
    time_lst.append(minute)
    prev_price, current_price, peak, valley = detector0.process_price(price, minute)

    logret = np.log(price / prev_price) if prev_price else 0
    kf_ca.predict()
    kf_ca.update(logret)
    kf_ca_saver.save()
    s = kf_ca_saver # shortcut
    logger.info(f"kf_ca {price:.2f} {logret:.2%} {np.log(price / base_price):.2%}")
    logger.info(f"x={s.x[-1:]}")
    logger.info(f"P={s.P[-1:]}")
    logger.info(f"K={s.K[-1:]}")
    logger.info(f"y={s.y[-1:]}")


In [ ]:
# After processing all prices, you can get the peaks and valleys
peaks = detector0.get_peaks()
valleys = detector0.get_valleys()


In [ ]:
base_time = datetime.datetime.strptime('9:30', '%H:%M')

def min_to_time(minute):
    return base_time + datetime.timedelta(minutes=int(minute - trading_start_time))

In [ ]:
# plot the prices and peaks and valleys
fig, ax = plt.subplots(figsize=(50, 6))
x = range(trading_start_time, trading_end_time)
# times = [datetime.time(minute // 60, minute % 60) for minute in time_lst]
times = [base_time + datetime.timedelta(minutes=int(i-trading_start_time)) for i in x]
ax.plot(times, price_lst2)
fig.autofmt_xdate()
ax.set_xlim(base_time - datetime.timedelta(minutes=2), times[-1] + datetime.timedelta(minutes=2))
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=5))  # Set ticks every 5 minutes
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%H:%M'))
for peak in detector0.get_peaks():
    ax.plot(min_to_time(peak[1]), peak[0], 'ro')
for valley in detector0.get_valleys():
    ax.plot(min_to_time(valley[1]), valley[0], 'bo')
# plt.plot(range(trading_start_time, trading_end_time), peaks, 'ro')
# plt.plot(range(trading_start_time, trading_end_time), valleys, 'bo')
plt.show()

In [ ]:
import pandas as pd
import numpy as np

class IntradayPeakValleyDetector:
    def __init__(self, window_size=5, lag=2):
        self.window_size = window_size
        self.lag = lag
        self.prices = []
        self.peaks = []
        self.valleys = []
        self.current_time = []

    def process_price(self, timestamp, price):
        self.current_time.append(timestamp)
        prev_price = self.prices[-1] if self.prices else None
        self.prices.append(price)
        peak = valley = None

        if len(self.prices) >= self.window_size:
            # mid = self.window_size // 2
            mid = self.window_size - self.lag
            window = self.prices[-self.window_size:]
            mid_time = self.current_time[-self.window_size + mid]

            if all(window[mid] > p for p in window[:mid]) and all(window[mid] > p for p in window[mid+1:]):
                peak = (window[mid], mid_time)
                self.peaks.append((window[mid], mid_time))

            elif all(window[mid] < p for p in window[:mid]) and all(window[mid] < p for p in window[mid+1:]):
                valley = (window[mid], mid_time)
                self.valleys.append((window[mid], mid_time))

            self.prices.pop(0)
            self.current_time.pop(0)

        return prev_price, price, peak, valley

    def get_peaks(self):
        return self.peaks
    
    def get_peaks_ts(self):
        return [(p, IntradayPeakValleyDetector.timestamp_to_minute(ts)) for p, ts in self.peaks]

    def get_valleys(self):
        return self.valleys
    
    def get_valleys_ts(self):
        return [(v, IntradayPeakValleyDetector.timestamp_to_minute(ts)) for v, ts in self.valleys]

    def get_results(self):
        return {
            'peaks': self.peaks,
            'valleys': self.valleys
        }

    # function to convert timestamp to minute
    @staticmethod
    def timestamp_to_minute(timestamp):
        return timestamp.hour * 60 + timestamp.minute


In [ ]:
# function to convert timestamp to minute
def timestamp_to_minute(timestamp):
    return timestamp.hour * 60 + timestamp.minute

In [ ]:
# Example usage
detector2 = IntradayPeakValleyDetector(3, 2)

# Simulate streaming prices (replace this with real-time data feed)
timestamps = pd.date_range(start='2025-02-06 09:30:00', end='2025-02-06 16:00:00', freq='1min')
# prices = np.random.randn(len(timestamps)).cumsum() + 100

price_lst3 = []
for timestamp, price in zip(timestamps, prices):
    price_lst3.append(price)
    detector2.process_price(timestamp, price)

# results = detector.get_results()

# print("Peaks:")
# for peak in results['peaks']:
#     print(f"Time: {peak[0]}, Price: {peak[1]:.2f}")

# print("\nValleys:")
# for valley in results['valleys']:
#     print(f"Time: {valley[0]}, Price: {valley[1]:.2f}")


In [ ]:
# plot the prices and peaks and valleys
fig, ax = plt.subplots(figsize=(50, 6))
lag = detector2.lag
ax.plot(range(trading_start_time, trading_end_time), price_lst3)
for peak in detector2.get_peaks():
    ax.plot(timestamp_to_minute(peak[1]), peak[0], 'ro')
for valley in detector2.get_valleys():
    ax.plot(timestamp_to_minute(valley[1]), valley[0], 'bo')
# plt.plot(range(trading_start_time, trading_end_time), peaks, 'ro')
# plt.plot(range(trading_start_time, trading_end_time), valleys, 'bo')
plt.show()

In [ ]:
# compare detector1 and detector2 peaks
print("Detector1 Peaks:", peaks)
print("Detector2 Peaks:", detector2.get_peaks_ts())
